# VKX Reader - Vakaros VKX Telemetry Log

Copyright 2026 Michael George (AKA Logiqx).

This file is part of [GPS Wizard](https://github.com/Logiqx/gps-wizard) and is distributed under the terms of the GNU General Public License.

GPS Wizard is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

GPS Wizard is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with GPS Wizard. If not, see <https://www.gnu.org/licenses/>.

In [1]:
import os
import sys
import time

import numpy as np

import unittest

from base_reader import BaseReader

## Main Class

In [2]:
PRIMARY_TELEMETRY_ID = 0x02

payloadSizes = {
    0x01: 32,
    PRIMARY_TELEMETRY_ID: 44,
    0x03: 20,
    0x04: 13,
    0x05: 17,
    0x06: 18,
    0x07: 12,
    0x08: 13,
    0x0A: 16,
    0x0B: 16,
    0x0C: 12,
    0x0E: 16,
    0x0F: 16,
    0x10: 12,
    0x20: 13,
    0x21: 52,
    0xFE: 2,
    0xFF: 7
}

class VkxReader(BaseReader):
    '''VKX file - Vakaros VKX Telemetry Log'''

    def __init__(self, filename):
        '''Basic init just records the filename'''

        super().__init__(filename)


    def load(self):
        '''Load file into memory'''

        with open(self.filename, "rb") as f:
            self.buffer = f.read()
            
            # The memoryview will be used to access the buffer without intermediate copying
            self.bufferView = memoryview(self.buffer)

            if len(self.buffer) > 0:
                self.readData()


    def readData(self):
        '''Read GNSS frames into memory'''

        fileSize = len(self.buffer)

        # Lean buffer will only contain the Position, Velocity, and Orientation
        leanBuffer = bytearray()

        numRecords = 0

        id, payload = self.readRecord()

        while self.bufferPtr < fileSize:
            id, payload = self.readRecord()
            
            if id == PRIMARY_TELEMETRY_ID:
                # Use += because it is faster than extend - see https://stackoverflow.com/q/40004517
                leanBuffer += payload
                numRecords += 1
                       
        if self.bufferPtr != fileSize:
            raise ValueError('File appears to be truncated')

        dtype = self.getDatatype()
        
        self.rawData = np.frombuffer(leanBuffer, dtype=dtype, count=numRecords)

        self.consumeRawData()

        self.summarise()


    def readRecord(self):
        '''Read VKX record from buffer'''

        offset = self.bufferPtr

        id = self.readBytes(1)[0]

        try:
            payloadSize = payloadSizes[id]
        except:
            raise ValueError('Unexpected id 0x{:x} - offset 0x{:08x}'.format(id, offset))

        payload = self.readBytes(payloadSize)

        return id, payload


    def getDatatype(self):
        '''Get the datatype of the GNSS frames'''

        # Position, Velocity, and Orientation - 44 bytes
        dtype = [
            ('utc_time', '<u8'),
            ('latitude', '<i4'),
            ('longitude', '<i4'),
            ('sog', '<f4'),
            ('cog', '<f4'),
            ('altitude', '<u2'),
            ('altitude_x', '<u2'),
            ('quaternion_w', '<f4'),
            ('quaternion_x', '<f4'),
            ('quaternion_y', '<f4'),
            ('quaternion_z', '<f4')
        ]

        return np.dtype(dtype)


    def consumeRawData(self):
        '''Consume data - convert to standard data structure'''

        # Create new track object
        track = self.addTrack()

        # Convert timestamp from ms to s
        track.data['ts'] = np.divide(self.rawData['utc_time'], 1000, dtype='float64')

        # Convert latitude and longitude to decimal
        track.data['lat'] = np.divide(self.rawData['latitude'], 10000000, dtype='float64')
        track.data['lon'] = np.divide(self.rawData['longitude'], 10000000, dtype='float64')

        # SOG is already in m/s
        track.data['sog'] = self.rawData['sog']

        # Convert COG from radians to degrees
        track.data['cog'] = np.degrees(self.rawData['cog'])

        # Ignore altitude because the values in the test file make no sense, perhaps two 16-bit floats?
        #print(np.unique(self.rawData['altitude']))

## Unit Tests

In [3]:
class TestVkxData(unittest.TestCase):
    '''Class to test Vakaros data was correctly loaded'''

    def testNumRecords(self):
        '''Test the number of records is as expected'''

        for fieldName in vkxReader.tracks[0].data:
            self.assertEqual(vkxReader.tracks[0].data[fieldName].size, 5111)


    def testTimestamp(self):
        '''Test the timestamp is as expected'''

        self.assertEqual(vkxReader.tracks[0].data['ts'].min(), np.float64(1781365877.056))
        self.assertEqual(vkxReader.tracks[0].data['ts'].max(), np.float64(1781368432.063))


    def testLat(self):
        '''Test the latitude is as expected'''

        self.assertEqual(vkxReader.tracks[0].data['lat'].min(), np.float64(50.5716237))
        self.assertEqual(vkxReader.tracks[0].data['lat'].max(), np.float64(50.5867736))


    def testLon(self):
        '''Test the longitude is as expected'''

        self.assertEqual(vkxReader.tracks[0].data['lon'].min(), np.float64(-2.461573))
        self.assertEqual(vkxReader.tracks[0].data['lon'].max(), np.float64(-2.4418115))


    def testSog(self):
        '''Test the speed over ground is as expected'''

        self.assertEqual(vkxReader.tracks[0].data['sog'].min(), np.float32(0.0))
        self.assertEqual(vkxReader.tracks[0].data['sog'].max(), np.float32(14.14723))


    def testCog(self):
        '''Test the course over ground is as expected'''

        self.assertEqual(vkxReader.tracks[0].data['cog'].min(), np.float32(0.0))
        self.assertEqual(vkxReader.tracks[0].data['cog'].max(), np.float32(360.0))

In [4]:
if __name__ == '__main__':
    projdir = os.path.realpath(os.path.join(sys.path[0], "..", ".."))

    filename = os.path.join(projdir, 'sessions', '20260613', 'Kuba_13-06-2026.vkx')
    vkxReader = VkxReader(filename)

    pc1 = time.perf_counter()
    vkxReader.load()
    pc2 = time.perf_counter()

    print("\nVKX files loaded in in %0.2f seconds" % (pc2 - pc1))


VKX files loaded in in 0.01 seconds


In [5]:
if __name__ == '__main__':
    # Determine whether session is interactive or batch to facilitate unittest.main(..., exit=testExit)
    import __main__ as main
    testExit = hasattr(main, '__file__')

    unittest.main(argv=['first-arg-is-ignored'], exit=testExit)

......
----------------------------------------------------------------------
Ran 6 tests in 0.004s

OK
